In [5]:
import torch
import numpy as np

## 1. 交叉熵losss
- 场景：一般用于计算分类损失，如LLM自回归时，输入为[1, 2, 3], 模型要输出下一个token 4，但是模型输出的是一个概率分布，比如vocab_size=10(0-9), 模型输出的是10个位置的概率，总和为1
当action被离散化成token时(比如RT-2， OpenVLA范式)，也是用此损失
- 输入: logits: Tensor[B, l , vocab_size], label: Tensor[B, l]
- 输出: loss: scalar

In [ ]:
# v1 似乎最后总和不为1啊
# def softmax(x : torch.Tensor , dim=-1):
#     return torch.exp(x) / torch.sum(x, dim=dim).unsqueeze(dim).expand(x.shape) 

# v2 似乎是因为sum里面忘了exp
def softmax(x : torch.Tensor , dim=-1):
    return torch.exp(x) / torch.sum(torch.exp(x), dim=dim).unsqueeze(dim).expand(x.shape) 

# V3 数值稳定写法，exp有溢出风险

In [53]:
def cross_entropy(logits: torch.Tensor, labels: torch.Tensor):
    """
    input:
        logis: [B, l, vocab_size]
        labels: [B, l]
    - log (label对应位置概率)
    output:
        loss: [B, ]
    """
    # 先要对 logits最后一维进行softmax
    logits = softmax(logits, dim=-1)

    batch_loss = [] # 放每个batch里所有步的概率
    for batch, label in zip(logits, labels):
        seq_prob = [] # 放每一句的
        for sequence, l in zip(batch, label):
            seq_prob.append(sequence[int(l)]) # 放进去这一步对应token 的logits
        # 计算这一句的loss
        seq_loss = torch.sum( - torch.log(torch.Tensor(seq_prob))) / len(seq_prob)
        batch_loss.append(seq_loss)
        
    return torch.stack(batch_loss)
    
    # 计算每个Batch的 loss


In [ ]:
# 输入输出评估
logits = torch.Tensor([
    [  # batch 0
        [2.0, 1.0, 0.0, -1.0],
        [0.0, 1.0, 2.0,  3.0],
        [1.0, 1.0, 1.0,  1.0],
    ],
    [  # batch 1
        [1.0, 2.0, 3.0,  4.0],
        [3.0, 1.0, 0.0, -1.0],
        [0.0, 0.0, 2.0,  0.0],
    ],
])

labels = torch.Tensor([
    [0, 2, 3],  # batch 0
    [3, 0, 2],  # batch 1
])
# batch_losses = [1.0888913, 0.3220417,]

loss = cross_entropy(logits, labels)
loss # ok 啊
# assert abs(cross_entropy(logits, labels) - 0.7054665) < 1e-6

tensor([1.0889, 0.3220])

In [ ]:
a = torch.Tensor([[1, 2], 
                  [4, 4]])
exp = torch.exp(a)
print(exp)
s = torch.sum(exp, dim=-1).unsqueeze(-1).expand(a.shape)
print(s)


tensor([[ 2.7183,  7.3891],
        [54.5981, 54.5981]])
tensor([[ 10.1073,  10.1073],
        [109.1963, 109.1963]])
